# Preparación, limpieza y estructuración de datos: churn en plataformas digitales

**Actividad:** GA1-220501094-01-AA1-EV07 · Preparación, limpieza y estructuración de datos del proyecto individual con Pandas

**Programa:** Tecnólogo en Análisis y Desarrollo de Software · CTMA – SENA, Regional Antioquia · Ficha 3223873

**Aprendiz:** Nicolás Muñoz Hernández

**Avance de este cuaderno:** Lección 2 (limpieza)

**Proyecto:** Análisis y predicción de cancelación de suscripciones (churn) en plataformas digitales

**Pregunta orientadora:** ¿cómo preparar y estructurar los datos del proyecto para obtener información confiable que permita responder la pregunta analítica?

**Pregunta analítica del proyecto:** ¿qué patrones de interacción, frecuencia de uso y nivel de soporte técnico en los primeros 30 días permiten predecir el riesgo de cancelación?

> **Alcance ajustado.** El diagnóstico de calidad (`data_quality.ipynb` y `reports/data_quality.md`) concluyó que ningún dataset trae información por ventana de 30 días. Por eso este cuaderno responde con las variables disponibles: uso, inicios de sesión, tickets y llamadas de soporte, y fallos de pago.

| Parte | Contenido |
|---|---|
| Preparación | Carga de los dos datasets y estado inicial |
| Lección 2 | Duplicados, nulos, tipos, estandarización, funciones, pipeline y validaciones |
| Lección 3 | Filtros, ordenamiento, agrupaciones, indicadores y tablas dinámicas |
| Lección 4 | Relación entre datasets, estructuración, validación y exportación a `data/processed/` |

## Preparación: carga de los datos y estado inicial

Los dos archivos originales están en `data/raw/` y **no se modifican**: toda la limpieza se hace sobre copias en memoria y los resultados se guardan en `data/processed/`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# El cuaderno vive en notebooks/, así que los datos están un nivel arriba.
RUTA = next((c for c in [Path('../data/raw'), Path('data/raw')] if (c / 'churn.csv').exists()), None)
assert RUTA is not None, 'No encuentro los CSV: deben estar en data/raw/ (ver README.md).'
RAIZ = RUTA.parent.parent
SALIDA = RAIZ / 'data' / 'processed'
SALIDA.mkdir(parents=True, exist_ok=True)

fmt = lambda n: f'{n:,.0f}'.replace(',', '.')      # 1234567 -> 1.234.567

try:
    import pyarrow
    print('pyarrow', pyarrow.__version__)
except ImportError:
    print('AVISO: falta pyarrow. Instálalo con  python -m pip install pyarrow  y reinicia el kernel.')
print('pandas', pd.__version__)
print('Datos leídos desde:', RUTA.resolve())

In [ ]:
sus_raw = pd.read_csv(RUTA / 'customer_subscription_churn_usage_patterns.csv')
telco_raw = pd.read_csv(RUTA / 'churn.csv')

print('Suscripción (SaaS):', sus_raw.shape)
print('Telco:             ', telco_raw.shape)
# Deberías ver (2800, 10) y (4250, 21)

### Estado inicial de los datos

Se define una función que resume la calidad de una tabla. Se usará de nuevo al final para comparar el antes y el después.

In [ ]:
def perfil_calidad(df, nombre, etapa):
    texto = df.select_dtypes(include=['object', 'string', 'category']).columns
    return {
        'dataset': nombre,
        'etapa': etapa,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'duplicados': int(df.duplicated().sum()),
        'columnas_texto_o_categoria': len(texto),
        'memoria_kb': round(df.memory_usage(deep=True).sum() / 1e3, 1),
    }

estado = [perfil_calidad(sus_raw, 'suscripción', '1 inicial'),
          perfil_calidad(telco_raw, 'telco', '1 inicial')]
pd.DataFrame(estado)

In [ ]:
print(sus_raw.dtypes.astype(str).to_dict())
print()
print(telco_raw.dtypes.astype(str).to_dict())

---
# Lección 2 · Limpieza y estandarización

Para cada problema hay tres opciones: **eliminar, corregir o conservar**. La decisión depende de qué significa el dato, no solo de que "se vea raro". Orden de trabajo: duplicados → nulos → tipos → texto → comprobaciones → pipeline → validación.

## 2.1 Duplicados

Se revisan los duplicados exactos y también los que se repiten **ignorando el identificador** (dos filas iguales con distinto `user_id` serían el mismo cliente registrado dos veces).

In [ ]:
def revisar_duplicados(df, id_col):
    return {
        'duplicados exactos': int(df.duplicated().sum()),
        'identificadores repetidos': int(df[id_col].duplicated().sum()),
        'filas iguales ignorando el identificador': int(df.drop(columns=id_col).duplicated().sum()),
    }

dup = pd.DataFrame({'suscripción': revisar_duplicados(sus_raw, 'user_id'),
                    'telco': revisar_duplicados(telco_raw, 'customer_id')})
dup

In [ ]:
total_dup = int(dup.values.sum())
display(Markdown(
    "**Decisión:** " + (
        "no hay duplicados de ninguno de los tres tipos, así que **no se elimina ninguna fila**. Aun así, la función `quitar_duplicados` se incluye en el pipeline para que la limpieza siga siendo válida si llegan datos nuevos."
        if total_dup == 0 else
        f"hay {total_dup} casos que se deben revisar antes de eliminar."
    )
))

## 2.2 Valores faltantes

Un valor faltante no siempre aparece como `NaN`: también puede venir como texto vacío, como la palabra `NA` o como un cero que no es real. Se revisan las tres formas.

In [ ]:
def revisar_faltantes(df):
    nulos = int(df.isna().sum().sum())
    texto = df.select_dtypes(include=['object', 'string'])
    vacios = int(sum(texto[c].str.strip().isin(['', 'NA', 'N/A', 'nan', 'NaN', 'null', 'None']).sum() for c in texto))
    return {'nulos (NaN)': nulos, 'texto vacío o "NA"': vacios}

pd.DataFrame({'suscripción': revisar_faltantes(sus_raw), 'telco': revisar_faltantes(telco_raw)})

**Ceros sospechosos.** En el diagnóstico se vio que hay clientes de Telco con 0 minutos diurnos pero con uso normal en las otras franjas. Se revisan con detalle.

In [ ]:
cero_dia = telco_raw[telco_raw['total_day_minutes'] == 0]
print('Clientes con 0 minutos diurnos:', len(cero_dia))
display(cero_dia[['customer_id', 'total_day_minutes', 'total_day_calls', 'total_day_charge',
                  'total_eve_minutes', 'total_night_minutes', 'churn']])

# Referencia: los ceros en minutos internacionales (clientes sin plan internacional) son normales
print('Clientes con 0 minutos internacionales:', int((telco_raw['total_intl_minutes'] == 0).sum()))

**Decisión sobre faltantes**

| Columna | Situación | Decisión |
|---|---|---|
| Todas las columnas | No hay nulos ni textos vacíos | No se imputa nada: rellenar datos que no faltan los alteraría |
| `total_day_minutes` (Telco) | 2 clientes con 0 minutos, 0 llamadas y 0 cargo diurno, pero con uso normal de noche y tarde | **Conservar y marcar** con una columna indicadora (`zero_day_usage`, se crea en la Lección 4). No se imputa ni se borra: son 2 de 4.250 y no se sabe si el cero es real |
| `total_intl_minutes` (Telco) | Los 0 corresponden a clientes sin llamadas internacionales | Conservar: es un valor real |

## 2.3 Tipos de dato

Los tipos se corrigen con `astype`, `to_datetime` y `Categorical`. Se reduce el tamaño de los enteros y se convierte `churn` a `0/1` en **ambos** datasets (en uno era `Yes/No` y en el otro `True/False`).

In [ ]:
# Comprobar antes de convertir: ¿los valores caben en los tipos más pequeños y las etiquetas son las esperadas?
print('churn suscripción:', sorted(sus_raw['churn'].unique()))
print('churn telco:      ', sorted(telco_raw['churn'].unique().tolist()))
print('plan_type:        ', sorted(sus_raw['plan_type'].unique()))
print('Máximos suscripción:', sus_raw[['user_id', 'monthly_fee', 'support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago']].max().to_dict())
print('Máximos telco:', telco_raw[['account_length', 'number_vmail_messages', 'total_day_calls', 'total_eve_calls', 'total_night_calls', 'total_intl_calls', 'number_customer_service_calls']].max().to_dict())

## 2.4 Estandarización de textos y formatos

- `plan_type`: se quitan espacios y se unifican las mayúsculas, y se convierte en categoría **ordenada** (Basic < Standard < Premium).
- `state`: mayúsculas sin espacios.
- `area_code`: llega como `area_code_510`; se deja solo el número (`510`).
- Nombres de columnas: minúsculas y sin espacios.

In [ ]:
print(sorted(telco_raw['area_code'].unique()))
print(telco_raw['state'].str.len().value_counts().to_dict())
print((sus_raw['plan_type'] != sus_raw['plan_type'].str.strip().str.title()).sum(), 'planes con texto por corregir')

## 2.5 Funciones reutilizables y pipeline de limpieza

Cada paso es una función pequeña que recibe una tabla y devuelve una tabla nueva (sin modificar la original). Se encadenan con `pipe()`, y después de cada paso se anota el estado de la tabla para tener un **registro de cambios**.

In [ ]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla, 'paso': paso, 'filas': len(df), 'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()), 'memoria_kb': round(df.memory_usage(deep=True).sum() / 1e3, 1),
    })
    return df

def estandarizar_columnas(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    return df

def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def limpiar_texto(df, columnas, caso='title'):
    df = df.copy()
    for c in columnas:
        s = df[c].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
        df[c] = getattr(s.str, caso)()
    return df

def tipos_suscripcion(df):
    df = df.copy()
    df['signup_date'] = pd.to_datetime(df['signup_date'], format='%Y-%m-%d')
    df['plan_type'] = pd.Categorical(df['plan_type'], categories=['Basic', 'Standard', 'Premium'], ordered=True)
    df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})
    return df.astype({'user_id': 'int16', 'monthly_fee': 'int16', 'support_tickets': 'int8',
                      'payment_failures': 'int8', 'tenure_months': 'int8',
                      'last_login_days_ago': 'int8', 'churn': 'int8'})

def tipos_telco(df):
    df = df.copy()
    df['area_code'] = df['area_code'].str.replace('area_code_', '', regex=False)
    return df.astype({'state': 'category', 'area_code': 'category', 'churn': 'int8',
                      'account_length': 'int16', 'number_vmail_messages': 'int16',
                      'total_day_calls': 'int16', 'total_eve_calls': 'int16',
                      'total_night_calls': 'int16', 'total_intl_calls': 'int16',
                      'number_customer_service_calls': 'int8'})

In [ ]:
registro = []   # se reinicia aquí para no duplicar filas si se vuelve a ejecutar la celda

sus = (sus_raw
    .pipe(anotar, 'suscripción', '0 original')
    .pipe(estandarizar_columnas)
    .pipe(quitar_duplicados)
    .pipe(anotar, 'suscripción', '1 sin duplicados')
    .pipe(limpiar_texto, ['plan_type', 'churn'], caso='title')
    .pipe(anotar, 'suscripción', '2 texto estandarizado')
    .pipe(tipos_suscripcion)
    .pipe(anotar, 'suscripción', '3 tipos corregidos'))

telco = (telco_raw
    .pipe(anotar, 'telco', '0 original')
    .pipe(estandarizar_columnas)
    .pipe(quitar_duplicados)
    .pipe(anotar, 'telco', '1 sin duplicados')
    .pipe(limpiar_texto, ['state'], caso='upper')
    .pipe(anotar, 'telco', '2 texto estandarizado')
    .pipe(tipos_telco)
    .pipe(anotar, 'telco', '3 tipos corregidos'))

informe_cambios = pd.DataFrame(registro)
informe_cambios

In [ ]:
ic = informe_cambios.set_index(['tabla', 'paso'])
ms0, ms3 = ic.loc[('suscripción', '0 original'), 'memoria_kb'], ic.loc[('suscripción', '3 tipos corregidos'), 'memoria_kb']
mt0, mt3 = ic.loc[('telco', '0 original'), 'memoria_kb'], ic.loc[('telco', '3 tipos corregidos'), 'memoria_kb']
display(Markdown(
    f"**Interpretación del registro de cambios:**\n\n"
    f"- Ningún paso cambió el número de filas ni de nulos: no había duplicados ni faltantes que eliminar.\n"
    f"- El único cambio de tamaño viene de los **tipos**: la tabla de suscripción pasó de {ms0:,.0f} a {ms3:,.0f} KB ({(1 - ms3 / ms0) * 100:.0f} % menos) y la de Telco de {mt0:,.0f} a {mt3:,.0f} KB ({(1 - mt3 / mt0) * 100:.0f} % menos)."
))

## 2.6 Validación de la limpieza

Las comprobaciones con `assert` detienen el cuaderno si algo no es como se espera. Así la limpieza queda verificada y no solo "a simple vista".

In [ ]:
def validar_suscripcion(df):
    assert df['user_id'].is_unique, 'user_id repetido'
    assert df.isna().sum().sum() == 0, 'hay nulos'
    assert df['churn'].isin([0, 1]).all(), 'churn fuera de 0/1'
    assert str(df['signup_date'].dtype).startswith('datetime64'), 'signup_date no es fecha'
    assert df['plan_type'].notna().all(), 'plan_type con valores desconocidos'
    assert df.groupby('plan_type', observed=True)['monthly_fee'].nunique().eq(1).all(), 'una tarifa por plan'
    assert df['avg_weekly_usage_hours'].between(0, 168).all(), 'horas de uso imposibles'
    assert df[['support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago']].ge(0).all().all(), 'valores negativos'
    return 'Suscripción: validación superada'

def validar_telco(df):
    assert df['customer_id'].is_unique, 'customer_id repetido'
    assert df.isna().sum().sum() == 0, 'hay nulos'
    assert df['churn'].isin([0, 1]).all(), 'churn fuera de 0/1'
    assert set(df['area_code'].unique()) == {'408', '415', '510'}, 'códigos de área inesperados'
    assert df['state'].str.len().eq(2).all(), 'estado con formato incorrecto'
    minutos = [c for c in df.columns if c.endswith('_minutes')]
    assert df[minutos].ge(0).all().all(), 'minutos negativos'
    assert df['number_customer_service_calls'].between(0, 20).all(), 'llamadas de soporte fuera de rango'
    return 'Telco: validación superada'

print(validar_suscripcion(sus))
print(validar_telco(telco))
print(sus.shape, telco.shape)
# Deberías ver (2800, 10) y (4250, 21), igual que al inicio: la limpieza no perdió registros

In [ ]:
# Los datos originales no se modificaron
assert sus_raw['churn'].isin(['Yes', 'No']).all() and telco_raw['churn'].dtype == bool
print('Los DataFrame originales (sus_raw, telco_raw) siguen intactos y data/raw/ no se tocó.')

estado += [perfil_calidad(sus, 'suscripción', '2 tras limpieza (L2)'),
           perfil_calidad(telco, 'telco', '2 tras limpieza (L2)')]
pd.DataFrame(estado)

In [ ]:
display(Markdown(
    "**Resultado de la Lección 2:** ambos datasets quedan **limpios y validados**. No se eliminaron registros ni se imputaron valores, porque no había duplicados ni nulos. "
    "Lo que sí se corrigió fue el formato: fechas reales, `churn` en 0/1 en las dos tablas, categorías ordenadas, `area_code` reducido al número y enteros del tamaño justo."
))

---
## Verificación del avance (Lección 2)

Antes de guardar: **Restart** y luego **Run All**. Todas las celdas deben ejecutarse sin error y las dos validaciones deben imprimir "validación superada".